# Lab 2 : What an AI agent is

*Week 5 · Utrains LLMOps 8 Week Course*

Run each cell from the top. Read what it prints before you run the next cell.

**(Reload this file from disk if it still looks long or complicated.)**


## Objective

In Lab 1 the model **asked** for a tool. Then we stopped.

This lab answers one question: **what happens next?**

An **AI agent** is simply:

1. The model **selects** a tool (name + arguments).
2. Your code **runs** that tool.
3. Your code **sends the tool result back** to the model.
4. The model writes the **final answer** for the user.

That is the whole idea. One path. One example.

**Before you start.** Same `week05/.env` with `ANTHROPIC_API_KEY`.


### Step 1. Load the model

Same as Lab 1.


In [ ]:
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()
llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
print("Model ready.")


### Step 2. One simple tool

We only need **one** tool for this lab: look up an order.

(You already practiced three tools in Lab 1. Here we keep the example tiny.)


In [ ]:
from langchain_core.tools import tool

ORDERS = {
    "ORD-1007": {"customer": "Riya Sharma", "item": "Laptop stand", "status": "shipped", "city": "Pune"},
}


@tool
def get_order_status(order_id: str) -> dict:
    """Look up one customer order by order id."""
    order = ORDERS.get(order_id)
    if order is None:
        return {"error": f"Order {order_id} was not found"}
    return {"order_id": order_id, **order}


# Store the tool in a variable (same idea as Lab 1's TOOLS list)
tool = get_order_status

print("Tool ready:", tool.name)
print("Sample data:", tool.invoke({"order_id": "ORD-1007"}))


### Step 3. The full agent path (one cell)

Read the prints in order:

1. **SELECT TOOL** — the model chooses `get_order_status`
2. **RUN TOOL** — your code runs it and gets real data
3. **SEND RESULT BACK** — you give that data to the model
4. **FINAL ANSWER** — the model writes a normal sentence for the user

Nothing else is happening.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# Give the model access to the tool from Step 2
llm_with_tools = llm.bind_tools([tool])

question = "What is the status of order ORD-1007?"
messages = [
    SystemMessage(content="You are an order-desk assistant. Use tools. Do not invent data."),
    HumanMessage(content=question),
]

# 1) SELECT TOOL — model asks for a tool (same idea as Lab 1)
ai = llm_with_tools.invoke(messages)
messages.append(ai)

print("1) SELECT TOOL")
print("   tool_calls:", ai.tool_calls)
print()

# 2) RUN TOOL — your code runs what the model asked for
call = ai.tool_calls[0]
result = tool.invoke(call["args"])

print("2) RUN TOOL")
print("   name:", call["name"])
print("   args:", call["args"])
print("   result:", result)
print()

# 3) SEND RESULT BACK — model can now read the tool output
messages.append(
    ToolMessage(content=str(result), tool_call_id=call["id"])
)

print("3) SEND RESULT BACK")
print("   (tool result added to the conversation)")
print()

# 4) FINAL ANSWER — call the model again
final = llm_with_tools.invoke(messages)

print("4) FINAL ANSWER")
print("  ", final.content)


## What you should remember

In one sentence:

**An agent = model selects a tool → your code runs it → you send the result back → model answers.**

Lab 1 stopped after “select a tool.”  
Lab 2 completed the path.

**Lab 3** will show when this still fails (missing data, no matching tool, guessing with no tools).
